In [1]:
# read it in to inspect it
with open('D:\Deepam\Projects\Building a GPT model from scratch\input.txt', encoding='utf-8') as f:
    text = f.read()

<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Deepam Shah\AppData\Local\Temp\ipykernel_17668\3739891599.py:2: SyntaxWarning: invalid escape sequence '\D'
  with open('D:\Deepam\Projects\Building a GPT model from scratch\input.txt', encoding='utf-8') as f:


In [3]:
print("Length of dataset in characters:", len(text))

Length of dataset in characters: 1115393


In [34]:
# setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [52]:
# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # What is the maximum context length for prediction
max_iters = 3000
eval_interest = 300
learning_rate = 1e-2
eval_iters = 200

In [53]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [54]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [55]:
# create a mapping from characters to integers
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [56]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earlier will go to the GPT look like this

torch.Size([1115393]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [57]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [58]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [59]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context} the target: {target}")

When input is tensor([18]) the target: 47
When input is tensor([18, 47]) the target: 56
When input is tensor([18, 47, 56]) the target: 57
When input is tensor([18, 47, 56, 57]) the target: 58
When input is tensor([18, 47, 56, 57, 58]) the target: 1
When input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [60]:
torch.manual_seed(1337)

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1: i+block_size+1] for i in ix])
    x, y =  x.to(device), y.to(device)
    return x, y

xb, yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print("----")

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")
        

inputs:
torch.Size([32, 8])
tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43],
        [57,  0, 58, 46, 39, 52,  1, 50],
        [26, 19,  1, 20, 17, 26, 30, 37],
        [43,  8,  0,  0, 31, 43, 41, 53],
        [54, 54, 63,  1, 40, 56, 43, 43],
        [13, 30, 24, 21, 31, 24, 17, 10],
        [39, 63,  1, 21, 11,  1, 47, 58],
        [51, 43,  1, 63, 53, 59,  1, 58],
        [ 1, 42, 59, 49, 43,  2,  1, 58],
        [ 1, 42, 47, 57, 58, 43, 51, 54],
        [43,  1, 58, 46, 53, 59,  1, 40],
        [39, 56, 58,  1, 58, 46, 53, 59],
        [30, 47, 41, 46, 51, 53, 52, 42],
        [58, 39, 47, 52,  1, 39,  1, 41],
        [43,  1, 61, 43, 39, 49, 43, 56],
        [42,  6,  1, 21,  1, 51, 59, 57],
        [46, 43,  1, 61, 53, 51, 43, 52],
        [56, 57,  1, 53, 59, 56,  1, 46],
        [31, 46, 39, 50, 50,  1, 21,  1],
        [ 1, 43, 39, 57, 43,  1, 46, 47],
      

In [68]:
@torch.inference_mode()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [69]:
print(xb) # our input to the transformer

tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43],
        [57,  0, 58, 46, 39, 52,  1, 50],
        [26, 19,  1, 20, 17, 26, 30, 37],
        [43,  8,  0,  0, 31, 43, 41, 53],
        [54, 54, 63,  1, 40, 56, 43, 43],
        [13, 30, 24, 21, 31, 24, 17, 10],
        [39, 63,  1, 21, 11,  1, 47, 58],
        [51, 43,  1, 63, 53, 59,  1, 58],
        [ 1, 42, 59, 49, 43,  2,  1, 58],
        [ 1, 42, 47, 57, 58, 43, 51, 54],
        [43,  1, 58, 46, 53, 59,  1, 40],
        [39, 56, 58,  1, 58, 46, 53, 59],
        [30, 47, 41, 46, 51, 53, 52, 42],
        [58, 39, 47, 52,  1, 39,  1, 41],
        [43,  1, 61, 43, 39, 49, 43, 56],
        [42,  6,  1, 21,  1, 51, 59, 57],
        [46, 43,  1, 61, 53, 51, 43, 52],
        [56, 57,  1, 53, 59, 56,  1, 46],
        [31, 46, 39, 50, 50,  1, 21,  1],
        [ 1, 43, 39, 57, 43,  1, 46, 47],
        [ 0, 32, 53,  1, 24, 53, 5

In [70]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1,:] # becomes (B,C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B,C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel(vocab_size)
m = model.to(device)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long).to(device), 
                        max_new_tokens=100)[0].tolist()))

torch.Size([256, 65])
tensor(4.8136, device='cuda:0', grad_fn=<NllLossBackward0>)

pYCXxfRkRZd
wc'wfNfT;OLlTEeC K
jxqPToTb?bXAUG:C-SGJO-33SM:C?YI3a
hs:LVXJFhXeNuwqhObxZ.tSVrddXlaSZaNe


In [71]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

In [76]:
batch_size = 32
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interest == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    
    # sample a batch of data
    xb, yb = get_batch("train")

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 2.4546, val loss 2.4879
step 300: train loss 2.4576, val loss 2.4835
step 600: train loss 2.4510, val loss 2.4913
step 900: train loss 2.4641, val loss 2.4985
step 1200: train loss 2.4556, val loss 2.4880
step 1500: train loss 2.4587, val loss 2.4807
step 1800: train loss 2.4623, val loss 2.4839
step 2100: train loss 2.4610, val loss 2.4912
step 2400: train loss 2.4644, val loss 2.4948
step 2700: train loss 2.4514, val loss 2.4783


In [78]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long).to(device),
                        max_new_tokens=400)[0].tolist()))


MPUESTh, af Pre?

WISo myr f-
LIERor,
SS&y ar adsal this ghesthidin cour ay aney Iry ts I fr y ce.
JOMim ton, bemary.
Yof 'Wour me?
Isora anghy t--pongeshthe ten.
Sand thot sulin s th llety od, wiourco ffepyotssththas l.
TAn.
Mourethal wave.
se ed Pe bene ovetour?
Casscher os cok heding.

O:
He d he be fe f tas ny, ct Clo gscest hes, n ldu he n, soxcharerean! beaker aghercobun wsam k s withoumas F
